In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import itertools
import time as tt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input, TimeDistributed, Conv2D, MaxPooling2D, Flatten
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.python.client import device_lib

In [ ]:
# GET CPU AND GPU
def get_available_devices():
    local_device_protos = device_lib.list_local_devices()
    return [x.name for x in local_device_protos]
    
# FUNCION RMSE
def rmse_loss(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))    
    
# ARRAY TO MATRIX
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - look_back - 1):
        a = dataset[i:(i + look_back), :]
        dataX.append(a)
        dataY.append(dataset[i + look_back, :])
    return np.array(dataX), np.array(dataY)

In [ ]:
# fix random seed for reproducibility
os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
random.seed(7)
np.random.seed(7)
tf.random.set_seed(7)
tf.config.experimental.enable_op_determinism()

In [ ]:
# DATASET LOADING
dataframe = pd.read_csv('/kaggle/input/datasets/andriuswinx/traffic-7x7-1/Traffic_7x7_1.csv', usecols=range(1,50), engine='python')
dataset_full = dataframe.values
dataset_full = dataset_full.astype('float32')
num_features = dataset_full.shape[1] #7x7=49
total_rows = dataset_full.shape[0]

In [ ]:
# Parameter tweak for Grid Search
hyperparameters = {
    'look_back': [12, 24],
    'batch_size': [20, 25],
    'n_epochs_update': [6, 7],
    'threshold': [0, 0.1],
    'num_filters': [16, 20]
}

combinations = list(itertools.product(*hyperparameters.values()))
output_csv_file = '/kaggle/working/GridSearch_CNNLSTM_repro2.csv'

In [ ]:
for combi in combinations:
    tf.keras.backend.clear_session()
    tf.random.set_seed(7)
    os.environ['PYTHONHASHSEED'] = '0'
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
    random.seed(7)
    np.random.seed(7)
    tf.config.experimental.enable_op_determinism()
    
    params = dict(zip(hyperparameters.keys(), combi))
    
    current_look_back = params['look_back']
    current_batch_size = params['batch_size']
    current_n_epochs_update = params['n_epochs_update']
    current_threshold = params['threshold']
    current_filters = params['num_filters']
    
    print(f"\n=======================================================")
    print(f"Testing CNN-LSTM: look_back={current_look_back}, batch_size={current_batch_size}, updates={current_n_epochs_update}, threshold={current_threshold}")
    
    # SEPARATE INTO TRAIN AND TEST (Dynamic Batch Divisibility Fix)
    desired_train_size = 400
    train_available_batches = (desired_train_size - current_look_back - 1) // current_batch_size
    train_size = current_batch_size * train_available_batches + current_look_back + 1
    
    available_batches = (total_rows - current_look_back - 1 - train_size) // current_batch_size
    test_size = current_batch_size * available_batches + current_look_back + 1
    
    dataset = dataset_full[0:train_size+test_size]

    scaler = MinMaxScaler(feature_range=(0, 1))
    train = dataset[0:train_size, :]
    test = dataset[train_size : train_size+test_size, :]
    
    train = scaler.fit_transform(train)
    test = scaler.transform(test)

    # X=[T,T+LOOKBACK]  Y=[T+LOOKBACK+1]
    look_back = current_look_back
    trainX, trainY = create_dataset(train, look_back)
    testX, testY = create_dataset(test, look_back)

    # CNN SPATIAL RESHAPE (Crucial for Conv2D)
    trainX = trainX.reshape((trainX.shape[0], look_back, 7, 7, 1))
    testX = testX.reshape((testX.shape[0], look_back, 7, 7, 1))

    # SET THE MODEL PARAMETERS
    batch_size = current_batch_size
    n_epochs = 400
    #num_filters = 20

    # CNN-LSTM MODEL (Keras 3 Compliant)
    myCNNLSTM = Sequential()
    myCNNLSTM.add(Input(batch_shape=(batch_size, look_back, 7, 7, 1)))
    myCNNLSTM.add(TimeDistributed(Conv2D(filters=current_filters, kernel_size=(3,3), padding='same', activation='relu')))
    myCNNLSTM.add(TimeDistributed(Flatten())) # Flatten to connect CNN to LSTM
    myCNNLSTM.add(LSTM(look_back, stateful=True))
    myCNNLSTM.add(Dense(num_features))
    myCNNLSTM.compile(loss='mean_squared_error', optimizer='adam')

    # TRAIN LSTM MODEL
    print('Fitting myCNNLSTM on verbose 0')
    for i in range(n_epochs):
        myCNNLSTM.fit(trainX, trainY, epochs=1, batch_size=batch_size, verbose=0, shuffle=True)
        # Keras 3 Safe Reset
        for layer in myCNNLSTM.layers:
            if hasattr(layer, 'reset_states'):
                layer.reset_states()

    # STATIC TEST
    train_Y_u = scaler.inverse_transform(trainY)
    test_Y_u = scaler.inverse_transform(testY)

    train_Yhat_s = myCNNLSTM.predict(trainX, batch_size=batch_size)
    train_Yhat_su = scaler.inverse_transform(train_Yhat_s)

    test_Yhat_s = myCNNLSTM.predict(testX, batch_size=batch_size)
    test_Yhat_su = scaler.inverse_transform(test_Yhat_s)

    # ONLINE LEARNING TEST
    n_epochs_update = current_n_epochs_update
    error_threshold = current_threshold
    rmse_norm = 1
    online_yhat_u = list()
    times = list()
    optimizer = tf.keras.optimizers.Adam()

    # Initialize the LSTM with the last train batch (chronological)
    online_X = trainX[-batch_size:]
    online_Y = trainY[-batch_size:]

    print('Applying SKIMA backpropagation')

    for i in range(testY.shape[0]-batch_size):
        if rmse_norm > error_threshold:
            st = tt.time()
            for j in range(n_epochs_update):
                with tf.GradientTape() as tape:
                    predictions = myCNNLSTM(online_X, training=True)
                    loss = rmse_loss(online_Y, predictions)
                
                gradients = tape.gradient(loss, myCNNLSTM.trainable_variables)
                adjusted_gradients = [g * (rmse_norm**2) for g in gradients]
                optimizer.apply_gradients(zip(adjusted_gradients, myCNNLSTM.trainable_variables))
                
                for layer in myCNNLSTM.layers:
                    if hasattr(layer, 'reset_states'):
                        layer.reset_states()
            
            et = tt.time()
            elapsed = et-st
            times.append(elapsed)
            
        newX, newY = testX[i:i+batch_size], testY[i:i+batch_size]
        
        online_yhat = myCNNLSTM.predict(newX, batch_size=batch_size)
        online_yhat = scaler.inverse_transform(online_yhat)
        newY_u = scaler.inverse_transform(newY)
        
        rmse_norm =((np.sqrt(np.mean(((newY_u[-1,:] - online_yhat[-1,:])/online_yhat[-1,:])**2))))
        online_yhat_u.append(online_yhat[-1])
        
        online_X = newX
        online_Y = newY

    test_Yhat_ou=np.array(online_yhat_u)
    times=np.array(times)

    # Calculate root mean squared error
    trainScore=np.empty(num_features)
    testScore=np.empty(num_features)
    testScore_inc=np.empty(num_features)

    for i in range(num_features):
        trainScore[i] = np.sqrt(mean_squared_error(train_Y_u[:,i], train_Yhat_su[:,i]))/np.std(train_Y_u[:,i])
        testScore[i] = np.sqrt(mean_squared_error(test_Y_u[:,i], test_Yhat_su[:,i]))/np.std(test_Y_u[:,i])
        testScore_inc[i] = np.sqrt(mean_squared_error(test_Y_u[batch_size-1:-1,i], test_Yhat_ou[:,i]))/np.std(test_Y_u[batch_size-1:-1,i])

    timeScore = np.mean(times)
    avg_train_score = np.mean(trainScore)
    avg_test_score = np.mean(testScore)
    avg_test_score_inc = np.mean(testScore_inc)
    
    current_result = {
        'look_back': current_look_back,
        'batch_size': current_batch_size,
        'updates': current_n_epochs_update,
        'threshold': current_threshold,
        'num_filters': current_filters,
        'Avg_Train_RNMSE': avg_train_score,
        'Avg_Test_RNMSE': avg_test_score,
        'Avg_Test_Inc_RNMSE': avg_test_score_inc,
        'Avg_Test_Inc_Time': timeScore
    }
    
    combi_df = pd.DataFrame([current_result])
    file_exists = os.path.isfile(output_csv_file)
    combi_df.to_csv(output_csv_file, mode='a', header=not file_exists, index=False)